# Machine Learning II - Project 3
## Veggie Image Dataset Model Building, Tuning, and Analysis
### Gwen Horzempa

I used a tutorial by Daniel Bourke to load in the images and create the data loaders [1].

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data.sampler import SubsetRandomSampler
import torch.optim
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

import torchvision as tv
from torchvision import datasets
import torchvision.transforms as T
import torchvision.models

import random
import multiprocessing
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import matplotlib.pyplot as plt
from tqdm import tqdm

from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot
from functools import partial
import os
import pathlib
import torch

from PIL import Image
from torch.utils.data import Dataset
from typing import Tuple, Dict, List

from torch.utils.data import DataLoader

## Getting Data from Zipfile

In [ ]:
import requests
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("data")
image_path = data_path / "images"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "archive.zip", "r") as zip_ref:
        print("Unzipping veggie data..") 
        zip_ref.extractall(image_path)

In [ ]:
# Setup train and testing paths
test_dir = "data/images/veggies/test"
val_dir = "data/images/veggies/validation"
train_dir = "data/images/veggies/train"

train_dir, test_dir, val_dir

## Transforming Images

In [ ]:
def get_transforms(rand_augment_magnitude):
    mean = (0.4688234031200409, 0.4635027050971985, 0.3433191776275635)
    std = (0.19028286635875702, 0.19179819524288177, 0.1834646761417389)

    # Define our transformations
    return {"train": T.Compose([
                T.Resize(256), 
                T.RandomCrop(224), # take a random part of the image
                T.RandomHorizontalFlip(0.5),
                T.RandAugment(num_ops=2, magnitude=rand_augment_magnitude, interpolation=T.InterpolationMode.BILINEAR,),
                T.ToTensor(),
                T.Normalize(mean, std),]),
            "valid": T.Compose([
                T.Resize(256),
                T.CenterCrop(224),
                T.ToTensor(),
                T.Normalize(mean, std),]),
            "test": T.Compose([
                T.Resize(256),
                T.CenterCrop(224),
                T.ToTensor(),
                T.Normalize(mean, std),]),}

*This is the function I used to get the per-channel means and standard deviations for the images*

In [ ]:
from tqdm import tqdm

def get_mean_std(dataloader):
    mean = 0.
    std = 0.
    total_batches = 0

    for data, _ in dataloader:
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)

        mean += data.mean(2).sum(0)
        std += data.std(2).sum(0)
        total_batches += batch_samples

    mean /= total_batches
    std /= total_batches

    return mean.tolist(), std.tolist()

train_mean, train_std = get_mean_std(train_dataloader)
val_mean, val_std = get_mean_std(val_dataloader)
print(f"Train mean: {train_mean}\n Train std: {train_std}")
print(f"Val mean: {val_mean}\n Val std: {val_mean}")

## Custom Data Loaders

In [ ]:
def get_data_loaders(valid_size, transforms, num_workers, random_seed=1971, batch_size=32):

    # Reseed random number generators to get a deterministic split. This is useful when comparing experiments, so you'll know they all run on the same data.
    # In principle you should repeat this a few times (cross validation) to see the variability of your measurements.
    torch.manual_seed(random_seed)
    random.seed(random_seed)
    np.random.seed(random_seed)

    test_dir = "data/images/veggies/test"
    val_dir = "data/images/veggies/validation"
    train_dir = "data/images/veggies/train"

    print(type(transforms))

    train_data = datasets.ImageFolder(root=train_dir, 
                                  transform=transforms['train']) # transforms to perform on data (images)

    val_data = datasets.ImageFolder(root=val_dir, 
                                  transform=transforms['valid']) 


    # Compute how many items we will reserve for the validation set
    n_tot = len(train_data)
    split = int(np.floor(valid_size * n_tot))

    # prepare data loaders (combine dataset and sampler)

    train_dataloader = DataLoader(dataset=train_data, 
                            batch_size=batch_size,
                            num_workers=num_workers, # how many subprocesses to use for data loading? (higher = more)
                            shuffle=True) 

    val_dataloader = DataLoader(dataset=val_data, 
                            batch_size=batch_size, # how many samples per batch?
                            num_workers=num_workers, # how many subprocesses to use for data loading? (higher = more)
                            shuffle=False) # shuffle the data?

    # Get the CIFAR10 test dataset from torchvision.datasets, set the transforms, and prepare the data loader.

    test_data = datasets.ImageFolder(root=test_dir, 
                                 transform=transforms['test'])

    test_dataloader = DataLoader(dataset=test_data, 
                             batch_size=batch_size, 
                             num_workers=num_workers) 

    return {'train': train_dataloader, 'valid': val_dataloader, 'test': test_dataloader}

In [ ]:
classes = [
    'Bean', 
    'Bitter_Gourd', 
    'Bottle_Gourd', 
    'Brinjal', 
    'Broccoli', 
    'Cabbage', 
    'Capsicum', 
    'Carrot', 
    'Cauliflower', 
    'Cucumber', 
    'Papaya', 
    'Potato', 
    'Pumpkin', 
    'Radish', 
    'Tomato']

# Model Building

## Model 1

In [ ]:
model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
n_classes = len(classes)
n_inputs = model.fc.in_features

# Feel free to experiment with more complicated heads
model.fc = nn.Linear(n_inputs, n_classes)

In [ ]:
frozen_parameters = []

for p in model.parameters():
    # Freeze only parameters that are not already frozen
    if p.requires_grad:
        p.requires_grad = False
        frozen_parameters.append(p)

print(f"Froze {len(frozen_parameters)} groups of parameters")

# Now let's thaw the parameters of the head we have added
for p in model.fc.parameters():
    p.requires_grad = True

## Model 2

In [ ]:
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model = nn.Sequential(
    DefaultConv2d(in_channels=3, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=256, out_channels=512), nn.ReLU(),
    DefaultConv2d(in_channels=512, out_channels=512), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=100352, out_features=128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=15))

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224))

In [ ]:
print(model)

In [ ]:
# specify loss function (categorical cross-entropy)
loss = nn.CrossEntropyLoss()

# specify optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min")

In [ ]:
n_workers = multiprocessing.cpu_count() # number of logical CPUs
print('Using {} logical CPUs.'.format(n_workers))

batch_size = 32 # training batch size
valid_size = 0.2 # proportion of downloaded training data to use for validation

transforms = get_transforms(rand_augment_magnitude=2)
print(type(transforms))
data_loaders = get_data_loaders(valid_size, transforms, n_workers, batch_size)

# Training

In [ ]:
def train_one_epoch(train_dataloader, model, optimizer, loss):
    """
    Performs one epoch of training
    """

    # Move model to GPU if available
    if torch.cuda.is_available():
        model.cuda()  # -

    # Set the model to training mode. Some layers perform differently depending
    # on whether or not you are training or evaluating, e.g., batch normalization
    # and dropout.
    model.train()

    # Loop over the training data
    train_loss = 0.0

    for batch_idx, (data, target) in tqdm(
        enumerate(train_dataloader),
        desc="Training",
        total=len(train_dataloader),
        leave=True,
        ncols=80,
    ):
        # move data to GPU if available
        if torch.cuda.is_available():
            data, target = data.cuda(), target.cuda()

        # 1. clear the gradients of all optimized variables
        optimizer.zero_grad()

        # 2. forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)

        # 3. calculate the loss
        loss_value = loss(output, target)

        # 4. backward pass: compute gradient of the loss with respect to model parameters
        loss_value.backward() # The .backward() method is built-in; we don't have to define it in our model.

        # 5. perform a single optimization step (parameter update)
        optimizer.step()

        # update average training loss
        train_loss = train_loss + (
            (1 / (batch_idx + 1)) * (loss_value.data.item() - train_loss))

    return train_loss

In [ ]:
def valid_one_epoch(valid_dataloader, model, loss):
    """
    Validate at the end of one epoch
    """

    # During validation we don't need to accumulate gradients
    with torch.no_grad():

        # Set the model to evaluation mode. Some layers perform differently depending
        # on whether or not you are training or evaluating, e.g., batch normalization
        # and dropout.
        model.eval()

        # If the GPU is available, move the model to the GPU
        if torch.cuda.is_available():
            model.cuda()

        # Loop over the validation dataset and accumulate the loss
        valid_loss = 0.0
        for batch_idx, (data, target) in tqdm(
            enumerate(valid_dataloader),
            desc="Validating",
            total=len(valid_dataloader),
            leave=True,
            ncols=80,
        ):
            # move data to GPU if available
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()

            # 1. forward pass: compute predicted outputs by passing inputs to the model
            output = model(data)

            # 2. calculate the loss
            loss_value = loss(output, target)

            # Calculate average validation loss
            valid_loss = valid_loss + (
                (1 / (batch_idx + 1)) * (loss_value.data.item() - valid_loss))

    return valid_loss

In [ ]:
def optimize(data_loaders, model, optimizer, loss, n_epochs, save_path, patience, scheduler, interactive_tracking=False):
    # initialize tracker for minimum validation loss
    if interactive_tracking:
        liveloss = PlotLosses()
    else:
        liveloss = None

    # Loop over the epochs and keep track of the minimum of the validation loss
    valid_loss_min = None
    logs = {}
    wait = 0

    for epoch in range(1, n_epochs + 1):

        train_loss = train_one_epoch(
            data_loaders["train"], model, optimizer, loss
        )

        valid_loss = valid_one_epoch(data_loaders["valid"], model, loss)

        # print training/validation statistics
        print(f"Epoch: {epoch} \tTraining Loss: {train_loss:.6f} \tValidation Loss: {valid_loss:.6f}")

        # If the validation loss decreases by more than 1%, save the model
        if valid_loss_min is None or ((valid_loss_min - valid_loss) / valid_loss_min > 0.01):
            print(f"New minimum validation loss: {valid_loss:.6f}. Saving model ...")

            # Save the weights to save_path
            torch.save(model.state_dict(), save_path)  # -

            valid_loss_min = valid_loss
            wait = 0
        else:
            wait +=1
            if wait >= patience:
                print("Early stopping!")
                break
            100352

        # Update learning rate, i.e., make a step in the learning rate scheduler
        scheduler.step(valid_loss)

        # Log the losses and the current learning rate
        if interactive_tracking:
            logs["loss"] = train_loss
            logs["val_loss"] = valid_loss
            logs["lr"] = optimizer.param_groups[0]["lr"]
            liveloss.update(logs)
            liveloss.send()

In [ ]:
print(len(data_loaders["train"].dataset))
print(len(data_loaders["valid"].dataset))

In [ ]:
n_epochs = 50

optimize(
    data_loaders,
    model,
    optimizer,
    loss,
    n_epochs,
    'veggie.pt',
    7,
    scheduler,
    interactive_tracking=True
)

# Testing

In [ ]:
def one_epoch_test(test_dataloader, model, loss):
    # monitor test loss and accuracy
    test_loss = 0.
    correct = 0.
    total = 0.

    # we do not need the gradients
    with torch.no_grad():

        # set the model to evaluation mode
        model.eval()  # -

        # if the GPU is available, move the model to the GPU
        if torch.cuda.is_available():
            model = model.cuda()

        # Loop over test dataset
        # We also accumulate predictions and targets so we can return them
        preds = []
        actuals = []

        for batch_idx, (data, target) in tqdm(
                enumerate(test_dataloader),
                desc='Testing',
                total=len(test_dataloader),
                leave=True,
                ncols=80
        ):
            # move data to GPU if available
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()

            # 1. forward pass: compute predicted outputs by passing inputs to the model
            logits = model(data)  # =

            # 2. calculate the loss
            loss_value = loss(logits, target).detach()  # =

            # 3. update average test loss
            test_loss = test_loss + ((1 / (batch_idx + 1)) * (loss_value.data.item() - test_loss))

            # 4. convert logits to predicted class
            # NOTE: the predicted class is the index of the max of the logits
            pred = logits.data.max(1, keepdim=True)[1]  # =

            # 5. compare predictions to true label
            correct += torch.sum(torch.squeeze(pred.eq(target.data.view_as(pred))).cpu())
            total += data.size(0)

            preds.extend(pred.data.cpu().numpy().squeeze())
            actuals.extend(target.data.view_as(pred).cpu().numpy().squeeze())

    print('Test Loss: {:.6f}\n'.format(test_loss))

    print('\nTest Accuracy: %2d%% (%2d/%2d)' % (100. * correct / total, correct, total))

    return test_loss, preds, actuals

In [ ]:
model.load_state_dict(torch.load('veggie.pt'))

In [ ]:
test_loss, preds, actuals = one_epoch_test(data_loaders['test'], model, loss)

### Visualization of Model Performance

In [ ]:
def plot_confusion_matrix(pred, truth, classes):

    gt = pd.Series(truth, name='Ground Truth')
    predicted = pd.Series(pred, name='Predicted')

    confusion_matrix = pd.crosstab(gt, predicted)
    confusion_matrix.index = classes
    confusion_matrix.columns = classes

    fig, sub = plt.subplots()
    with sns.plotting_context("notebook"):

        ax = sns.heatmap(
            confusion_matrix,
            annot=True,
            fmt='d',
            ax=sub,
            linewidths=0.5,
            linecolor='lightgray',
            cbar=False
        )
        ax.set_xlabel("truth")
        ax.set_ylabel("pred")

    return confusion_matrix

In [ ]:
cm = plot_confusion_matrix(preds, actuals, classes)